In [1]:
import os
import torch
import time
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

In [2]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
import albumentations as A


class AugmentDataset(Dataset):
    def __init__(self, csv_path, image_dir, mask_dir, split='train', augment=True):
        self.metadata = pd.read_csv(csv_path)
        self.metadata = self.metadata[self.metadata['split'] == split].reset_index(drop=True)

        self.img_dir = image_dir
        self.msk_dir = mask_dir
        self.use_augment = augment

    def randomHorizontalFlip(self, img, msk, u=0.5):
        if np.random.rand() < u:
            img = cv2.flip(img, 1)
            msk = cv2.flip(msk, 1)
        return img, msk

    def randomVerticalFlip(self, img, msk, u=0.5):
        if np.random.rand() < u:
            img = cv2.flip(img, 0)
            msk = cv2.flip(msk, 0)
        return img, msk

    def randomRotate90(self, img, msk, u=0.5):
        if np.random.rand() < u:
            img = np.rot90(img).copy()
            msk = np.rot90(msk).copy()
        return img, msk

    def randomHueSaturationValue(
        self,
        img,
        hue_shift_limit=(-30, 30),
        sat_shift_limit=(-5, 5),
        val_shift_limit=(-15, 15),
        u=0.5,
    ):
        if np.random.rand() < u:
            hsv_img = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)

            hue, sat, val = cv2.split(hsv_img)

            hue = hue.astype(np.float32)
            sat = sat.astype(np.float32)
            val = val.astype(np.float32)

            hue += np.random.randint(hue_shift_limit[0], hue_shift_limit[1] + 1)
            sat += np.random.uniform(sat_shift_limit[0], sat_shift_limit[1])
            val += np.random.uniform(val_shift_limit[0], val_shift_limit[1])

            hue = np.clip(hue, 0, 179).astype(np.uint8)
            sat = np.clip(sat, 0, 255).astype(np.uint8)
            val = np.clip(val, 0, 255).astype(np.uint8)

            hsv_img = cv2.merge((hue, sat, val))
            img = cv2.cvtColor(hsv_img, cv2.COLOR_HSV2RGB)

        return img

    def randomShiftScaleRotate(
        self,
        img,
        msk,
        shift_limit=(-0.1, 0.1),
        scale_limit=(-0.1, 0.1),
        rotate_limit=(0, 0),
        aspect_limit=(-0.1, 0.1),
        borderMode=cv2.BORDER_CONSTANT,
        u=0.5,
    ):
        if np.random.rand() < u:
            h, w = img.shape[:2]

            rot_angle = np.random.uniform(*rotate_limit)
            scale_factor = np.random.uniform(1 + scale_limit[0], 1 + scale_limit[1])
            aspect_ratio = np.random.uniform(1 + aspect_limit[0], 1 + aspect_limit[1])

            scale_x = scale_factor * aspect_ratio / (aspect_ratio ** 0.5)
            scale_y = scale_factor / (aspect_ratio ** 0.5)

            shift_x = int(np.random.uniform(*shift_limit) * w)
            shift_y = int(np.random.uniform(*shift_limit) * h)

            cos_theta = np.cos(np.radians(rot_angle)) * scale_x
            sin_theta = np.sin(np.radians(rot_angle)) * scale_y

            rot_matrix = np.array([[cos_theta, -sin_theta],
                                   [sin_theta, cos_theta]])

            corners = np.array(
                [[0, 0], [w, 0], [w, h], [0, h]],
                dtype=np.float32
            )

            corners -= np.array([w / 2, h / 2], dtype=np.float32)

            transformed = np.dot(corners, rot_matrix.T) + np.array(
                [w / 2 + shift_x, h / 2 + shift_y],
                dtype=np.float32
            )

            dst = np.array(
                [[0, 0], [w, 0], [w, h], [0, h]],
                dtype=np.float32
            )

            transform = cv2.getPerspectiveTransform(
                transformed.astype(np.float32),
                dst
            )

            img = cv2.warpPerspective(
                img,
                transform,
                (w, h),
                flags=cv2.INTER_LINEAR,
                borderMode=borderMode,
                borderValue=(0, 0, 0),
            )

            msk = cv2.warpPerspective(
                msk,
                transform,
                (w, h),
                flags=cv2.INTER_NEAREST,
                borderMode=borderMode,
                borderValue=(0,),
            )

        return img, msk

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, index):
        sample = self.metadata.iloc[index]

        image_path = os.path.join(self.img_dir, sample['filename'])
        label_path = os.path.join(self.msk_dir, sample['maskname'])

        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        msk = cv2.imread(label_path, cv2.IMREAD_GRAYSCALE)

        img = cv2.resize(img, (512, 512), interpolation=cv2.INTER_LINEAR)
        msk = cv2.resize(msk, (512, 512), interpolation=cv2.INTER_NEAREST)

        if self.use_augment:
            img = self.randomHueSaturationValue(img)
            img, msk = self.randomShiftScaleRotate(img, msk)
            img, msk = self.randomHorizontalFlip(img, msk)
            img, msk = self.randomVerticalFlip(img, msk)
            img, msk = self.randomRotate90(img, msk)

        img = img.astype(np.float32) / 255.0
        img = img * 3.2 - 1.6
        img = np.transpose(img, (2, 0, 1))

        msk = msk.astype(np.float32) / 255.0
        msk = np.expand_dims(msk, axis=0)
        msk = (msk > 0.5).astype(np.float32)

        return torch.tensor(img), torch.tensor(msk)

In [3]:
import torch.nn.functional as F

In [4]:
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

def get_optimizer(model, lr=5e-4, weight_decay=1e-4):
    return AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

def get_scheduler(optimizer, T_0=10, T_mult=2, eta_min=1e-6):
    return CosineAnnealingWarmRestarts(
    optimizer, 
    T_0=T_0, 
    T_mult=T_mult, 
    eta_min=eta_min
)

In [5]:
def bce_dice_loss(y_pred_logits, y_true, smooth=1e-5):
    """
    y_pred_logits: Raw model outputs (NO Sigmoid applied in the architecture)
    y_true: Ground truth binary masks
    """

    logits = y_pred_logits.logits
    logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

    # 1. Numerically stable BCE using Logits
    bce = F.binary_cross_entropy_with_logits(logits, y_true)

    # 2. Apply Sigmoid safely inside the loss for Dice calculation
    y_pred_probs = torch.sigmoid(logits)

    # 3. Flatten the ENTIRE batch into a single 1D vector for maximum GPU speed
    y_pred_flat = y_pred_probs.view(-1)
    y_true_flat = y_true.view(-1)

    # 4. Global Dice calculation
    intersection = (y_pred_flat * y_true_flat).sum()
    dice = (2. * intersection + smooth) / (y_pred_flat.sum() + y_true_flat.sum() + smooth)

    dice_loss = 1.0 - dice

    # Combined loss
    return bce + dice_loss

In [6]:
import torch
import os

def save_model(model, path, epoch, score):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    state = {
        'model_state_dict': model.state_dict(),
        'epoch': epoch,
        'score': score
    }
    torch.save(state, path)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [8]:
# === Cấu hình ===
batch_size = 8
num_epochs = 41
lr = 5e-5
log_dir = '/kaggle/working/logs'
model_path = '/kaggle/working/best_model.pth'
csv_path = '/kaggle/input/datasets/vuongtran11233/new-division/data_division.csv'
image_dir = '/kaggle/input/datasets/vuongtran11233/combined-dataset/combined_dataset/images'
mask_dir = '/kaggle/input/datasets/vuongtran11233/combined-dataset/combined_dataset/masks'

In [9]:
# === Dataloader ===
train_dataset = AugmentDataset(csv_path, image_dir, mask_dir, split='train', augment=True)
val_dataset = AugmentDataset(csv_path, image_dir, mask_dir, split='val', augment=False)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4,pin_memory=True,drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4,pin_memory=True,drop_last=True)

In [10]:
import torch
from transformers import SegformerForSemanticSegmentation

# Initialize pre-trained SegFormer-B2
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b2",
    num_labels=1,
    ignore_mismatched_sizes=True
)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/99.0M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/364 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b2
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.bias                               | UNEXPECTED | 
classifier.weight                             | UNEXPECTED | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different ta

In [12]:
from collections import OrderedDict

# 2. Load your Kaggle .pth checkpoint
input_model_path = '/kaggle/input/models/vngtrnml17/segformer-b2-v1/pytorch/default/1/best_segformer.pth'
checkpoint = torch.load(input_model_path, map_location=device)

# 3. Handle the dictionary check (depending on how you saved it)
# If you didn't save it with a 'model_state_dict' wrapper, just use checkpoint directly
if 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
else:
    state_dict = checkpoint

# 4. Strip the 'module.' prefix from DataParallel
clean_state_dict = OrderedDict()
for key, value in state_dict.items():
    if key.startswith('module.'):
        clean_state_dict[key[7:]] = value
    else:
        clean_state_dict[key] = value

# 5. Load the weights into the model
# CRITICAL: strict=False allows it to ignore mismatched head weights smoothly
missing_keys, unexpected_keys = model.load_state_dict(clean_state_dict, strict=False)

# Optional: Print to ensure everything loaded smoothly except the classification head
print("Missing keys:", missing_keys)
print("Unexpected keys:", unexpected_keys)

model.to(device)

model.safetensors:   0%|          | 0.00/98.9M [00:00<?, ?B/s]

Missing keys: []
Unexpected keys: []


SegformerForSemanticSegmentation(
  (segformer): SegformerModel(
    (encoder): SegformerEncoder(
      (patch_embeddings): ModuleList(
        (0): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
          (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        )
        (1): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        )
        (2): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(128, 320, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (3): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(320, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)

In [13]:
# === Model, Loss, Optimizer ===
#model = DeepLabV3PlusResNet34(num_classes=1).to(device)
'''if torch.cuda.device_count() > 1:
    print(f"🚀 Using {torch.cuda.device_count()} GPUs for parallel training!")
    model = nn.DataParallel(model)'''
model.to(device)
optimizer = get_optimizer(model, lr)
scheduler = get_scheduler(optimizer)
EarlyStopPatience = 15

In [14]:
import gc

In [15]:
import csv
import os

# Create a custom log file in the output directory
log_file_path = "/kaggle/working/training_history.csv"

# Write headers if file doesn't exist
if not os.path.exists(log_file_path):
    with open(log_file_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "val_loss"])

# Inside your training loop at the end of every epoch:
def append_log(epoch, train_loss, val_loss):
    with open(log_file_path, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([epoch, train_loss, val_loss])

In [ ]:
# === Training Loop ===
best_loss = 999.0
for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    model.train()
    train_loss = 0

    for imgs, masks in tqdm(train_loader, desc='Training'):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = bce_dice_loss(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # === Validation ===
    model.eval()
    val_loss = 0
    all_preds = []
    all_masks = []

    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc='Validation'):
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            loss = bce_dice_loss(outputs, masks)
            val_loss += loss.item()

    val_loss /= len(val_loader)
    

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    append_log(epoch + 1, train_loss, val_loss)

    # === Scheduler + EarlyStopping ===
    scheduler.step(val_loss)
    
    if val_loss < best_loss:
        best_loss = val_loss
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        save_model(model, model_path, epoch, best_loss)
        patience_counter = 0  # Reset counter since we improved
        print("Validation loss improved!")
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{EarlyStopPatience}")

    # Trigger early stopping
    if patience_counter >= EarlyStopPatience:
        print("Early stopping triggered. Stopping training!")
        break

    del outputs, loss
    gc.collect()
    torch.cuda.empty_cache()

print("Training completed!")